# Implicit modelling

In this exercise we will work with the categorical variables in the Jura dataset.

In [ ]:
%%capture
!pip install cmcrameri # scientific color maps
!pip install git+https://github.com/italo-goncalves/geoML.git@claude

In [ ]:
import matplotlib.pyplot as plt
from matplotlib.colors import rgb2hex
import numpy as np
from cmcrameri import cm
import pandas as pd
import seaborn as sns

import geoml

import geoml.kernels as kr
import geoml.transform as tr
import geoml.latent as gl
import geoml.likelihood as lk

In [ ]:
jura_train, jura_val = geoml.datasets.jura()
jura_train

Let us start with the `Landuse` variable. First we check the number of categories:

In [ ]:
jura_train.variables['Landuse'].labels

Let's see the data locations. The Seaborn package will make this easy after converting the data to a data frame.

In [ ]:
jura_df = jura_train.as_data_frame()
jura_df

In [ ]:
rgb_mat = cm.roma(np.linspace(0, 1, len(jura_train.variables['Landuse'].labels)))
landuse_colors = [rgb2hex(c) for c in rgb_mat]


fig, ax = plt.subplots(figsize=(10, 10))

sns.scatterplot(data=jura_df, x='Xloc', y='Yloc', hue='Landuse_measurements_a', ax=ax,
                hue_order=jura_train.variables['Landuse'].labels,
                palette=landuse_colors)

ax.set_aspect('equal')
ax.set_xlabel('X (km)')
ax.set_ylabel('Y (km)')
fig.show()

## VGP model

VGP has three main components: a **latent variable network**, a **likelihood function**, and the **inducing points**. In this case we can use the data points themselves as inducing points.



In [ ]:
# A simple, stationary latent network
network_input = gl.BasicInput(
    inducing_points=jura_train,
    transform=tr.Isotropic(0.5)
)
network_output = gl.BasicGP(
    parent=network_input,
    size=len(jura_train.variables['Landuse'].labels),
    kernel=kr.Matern32(),
    fix_range=True
)

# The likelihood is chosen according to the
# variable type
landuse_lik = lk.CategoricalGaussianIndicator(
    n_components=len(jura_train.variables['Landuse'].labels)
)

# The VGP model
model = geoml.models.VGPNetwork(
    data=jura_train,
    variables='Landuse',
    latent_network=network_output,
    likelihoods=landuse_lik
)

# All data points used for training simultaneously
model.train_full(500)

plt.plot(model.training_log)

Creating an empy grid to receive the predictions:

In [ ]:
grid = geoml.data.Grid2D(
    start=[0, 0],
    end=[6, 6],
    n=[501, 501]
)

In [ ]:
model.predict(grid)

In [ ]:
# VGP outputs
pred = grid.get('Landuse/predicted').as_image()
uncertainty = grid.get('Landuse/uncertainty').as_image()

# plotting helpers
label_dict = {l: i for i, l in enumerate(jura_train.variables['Landuse'].labels)}
pred_num = np.vectorize(label_dict.get)(pred)

fig, ax = plt.subplots(1, 3, figsize=(15, 10), sharex=True, sharey=True)

sns.scatterplot(data=jura_df, x='Xloc', y='Yloc', hue='Landuse_measurements_a', ax=ax[0],
                hue_order=jura_train.variables['Landuse'].labels,
                palette=landuse_colors)

ax[0].set_aspect('equal')
ax[0].set_xlabel('X (km)')
ax[0].set_ylabel('Y (km)')
ax[0].set_title('Data')

ax[1].imshow(pred_num, cmap=cm.roma,
             vmin=0, vmax=len(landuse_colors)-1,
             extent=[0, 6, 0, 6], origin='lower')
ax[1].set_aspect('equal')
ax[1].set_xlabel('X (km)')
ax[1].set_ylabel('Y (km)')
ax[1].set_title('Predictions')

ax[2].imshow(uncertainty, cmap=cm.nuuk,
             vmin=0, vmax=1,
             extent=[0, 6, 0, 6], origin='lower')
ax[2].set_aspect('equal')
ax[2].set_xlabel('X (km)')
ax[2].set_ylabel('Y (km)')
ax[2].set_title('Uncertainty')

fig.show()

The uncertainty metric can be used to define a region of confidence for the model.

It can be seen that a stationary model struggles to adapt to the small details in the data.

In [ ]:
threshold = 0.75

fig, ax = plt.subplots(figsize=(10, 10))

ax.imshow(np.where(uncertainty < threshold, pred_num, np.nan),
          cmap=cm.roma,
          vmin=0, vmax=len(landuse_colors)-1,
          extent=[0, 6, 0, 6], origin='lower')

sns.scatterplot(data=jura_df, x='Xloc', y='Yloc', hue='Landuse_measurements_a', ax=ax,
                hue_order=jura_train.variables['Landuse'].labels,
                palette=landuse_colors)

ax.set_aspect('equal')
ax.set_xlabel('X (km)')
ax.set_ylabel('Y (km)')
fig.show()

## Rock data

Try making a model for the `Rock` variable.